*— cell 0 —*

# T05 Arm 2B (PageIndex) — Azure GPU run

Vectorless, reasoning-based retrieval. LLaMA 3.1 8B navigates a deterministic
ToC tree built locally from the AzureDI dump.

**Sequence:** GPU check → install Ollama → install pip deps → download
bundle → pre-flight smokes → time-budget gate → full run → upload
results → cleanup.

`run_subset` is **idempotent on `out_dir`** — restarting the kernel
mid-run resumes from any per-query files already written, locally or
from the blob.


In [2]:
# === cell 1 ===
# Edit only the two lines under "PER-RUN CONFIG" between launches.
# Everything else is derived.

# ── PER-RUN CONFIG ──────────────────────────────────────────────────────────
# One run = one PDF. Pick the BSARD stem from RQ2_T00_ORCHESTRATOR/data/selected_pdfs.json.
# RUN_NAME conventionally is "t04_<stem>" — matches the GT filename and the
# bundle name produced by scripts/prepare_azure_bundle.py.
#RUN_NAME = "t04_1804_03_21_1804032150"   # sanity-check PDF for the num_ctx/prompt fixes
#RUN_NAME = "t04_1867_06_08_1867060850"
#RUN_NAME = "t04_1967_10_10_1967101056"
#RUN_NAME = "t04_1967_10_10_1967101055"
RUN_NAME = "t04_2003_07_17_2013A31614"

# Container-level SAS with read+write+list+create. Rotate per session.
AZURE_CONTAINER_SAS_URL = ""  # paste your container SAS URL (read+write+list) before running

# ── REPO + AUTH ─────────────────────────────────────────────────────────────
GITHUB_TOKEN = ""                                # PAT with read scope
GITHUB_OWNER = "MariusPasch"
MONO_REPO = "bsard-rag-thesis"
GITHUB_BRANCH = "main"

# T01 (LLMClient), T02 (RetrievalResult + Article), T05 (this arm). T03+T04 are
# local-only deps used by build_tree.py — the tree is shipped pre-built in the
# bundle so we don't need them here.
SIBLING_REPOS = {
    "T01_SHARED":      "RQ2_T01_SHARED",
    "T02_DATA_LOADER": "RQ2_T02_DATA_LOADER",
    "T05_PAGEINDEX":   "RQ2_T05_ARM2_PAGEINDEX",
}

# ── BLOB LAYOUT (derived) ───────────────────────────────────────────────────
# Convention shared with notebooks/local_t05_eval_and_compare.ipynb:
#   t05_azure_bundles/<RUN_NAME>.zip   — uploaded by scripts/upload_to_blob.py
#   t05_results/<RUN_NAME>/q*.json     — written by section 9 below
BUNDLE_BLOB_NAME = f"t05_azure_bundles/{RUN_NAME}.zip"
RESULTS_BLOB_PREFIX = f"t05_results/{RUN_NAME}"

# ── LOCAL PATHS ON THE VM ───────────────────────────────────────────────────
REPOS_DIR = "/home/azureuser/repos"
BUNDLE_DIR = "/home/azureuser/bundle"
RESULTS_DIR = "/home/azureuser/results"
OLLAMA_MODEL = "llama3.1:8b"

print(f"RUN_NAME            {RUN_NAME}")
print(f"BUNDLE_BLOB_NAME    {BUNDLE_BLOB_NAME}")
print(f"RESULTS_BLOB_PREFIX {RESULTS_BLOB_PREFIX}")


RUN_NAME            t04_2003_07_17_2013A31614
BUNDLE_BLOB_NAME    t05_azure_bundles/t04_2003_07_17_2013A31614.zip
RESULTS_BLOB_PREFIX t05_results/t04_2003_07_17_2013A31614


*— cell 2 —*

## 1. GPU sanity check\n\nBail early if there is no CUDA device — the run is hours of CPU.

In [3]:
# === cell 3 ===
!nvidia-smi


Sat May 30 14:57:00 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.274.02             Driver Version: 535.274.02   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       On  | 00000001:00:00.0 Off |                  Off |
| N/A   31C    P8               9W /  70W |      2MiB / 16384MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

*— cell 4 —*

## 2. Ollama install / start / pull

Idempotent: skips install if already present, skips start if 11434 is
serving, skips pull if the model layer is cached.

In [4]:
# === cell 5 ===
import os
import subprocess
import time
import urllib.request

# Install Ollama if missing.
which = subprocess.run(["which", "ollama"], capture_output=True, text=True)
if which.returncode != 0:
    print("Installing Ollama...")
    subprocess.run(
        "curl -fsSL https://ollama.com/install.sh | sh",
        shell=True, check=True,
    )

# Push the GPU env vars into ollama serve's environment.
os.environ["OLLAMA_NUM_GPU"] = "99"
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
# KV-cache in 8-bit halves VRAM use at 16k context (Llama 3.1 8B's KV
# cache otherwise grows past the T4's 16 GB at num_ctx=16384). Has a
# negligible quality impact on this navigation task.
os.environ["OLLAMA_KV_CACHE_TYPE"] = "q8_0"
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"


def _ollama_up(url: str = "http://localhost:11434/api/tags") -> bool:
    try:
        with urllib.request.urlopen(url, timeout=2) as r:
            return r.status == 200
    except Exception:
        return False


if not _ollama_up():
    print("Starting ollama serve...")
    subprocess.Popen(
        ["bash", "-c", "ollama serve > /tmp/ollama.log 2>&1 &"],
    )
    for _ in range(30):
        if _ollama_up():
            break
        time.sleep(1)
    else:
        raise RuntimeError(
            "ollama serve did not become reachable on 11434 in 30s. "
            "Check /tmp/ollama.log."
        )

print("Pulling model (no-op if cached)...")
subprocess.run(["ollama", "pull", OLLAMA_MODEL], check=True)
print("Ollama ready.")


Pulling model (no-op if cached)...


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ 

Ollama ready.


pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling 667b0c1932bc: 100% ▕██████████████████▏ 4.9 GB                         
pulling 948af2743fc7: 100% ▕██████████████████▏ 1.5 KB                         
pulling 0ba8f0e314b4: 100% ▕██████████████████▏  12 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 455f34728c9b: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 


*— cell 6 —*

## 3. Clone sibling repos

Clones the three required GitHub repos into separate directories under
`REPOS_DIR`. Validates that any pre-existing directory's `origin` matches
the configured URL - mismatches abort loudly rather than silently
pulling from the wrong remote (a previous bug).

In [5]:
# === cell 7 ===
import subprocess
from pathlib import Path

# The former sibling repos (T01, T02, T05) are now subfolders of the single
# mono-repo, under RQ2_Structure_Aware_Retrieval/. Clone the mono-repo once.
MONO_DIR = Path(REPOS_DIR) / MONO_REPO
auth_url = (
    f"https://{GITHUB_TOKEN}@github.com/{GITHUB_OWNER}/{MONO_REPO}.git"
    if GITHUB_TOKEN else f"https://github.com/{GITHUB_OWNER}/{MONO_REPO}.git"
)
Path(REPOS_DIR).mkdir(parents=True, exist_ok=True)
if MONO_DIR.exists():
    subprocess.run(["git", "-C", str(MONO_DIR), "remote", "set-url", "origin", auth_url], check=True)
    subprocess.run(["git", "-C", str(MONO_DIR), "fetch", "--quiet"], check=True)
    subprocess.run(["git", "-C", str(MONO_DIR), "checkout", GITHUB_BRANCH], check=True)
    subprocess.run(["git", "-C", str(MONO_DIR), "pull", "--quiet"], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", "--branch", GITHUB_BRANCH, auth_url, str(MONO_DIR)], check=True)

# Each former sibling repo is now a subfolder of the mono-repo.
RQ2_ROOT = MONO_DIR / "RQ2_Structure_Aware_Retrieval"
repo_paths = {label: RQ2_ROOT / repo_name for label, repo_name in SIBLING_REPOS.items()}
for label, path in repo_paths.items():
    print(f"  {label:<18}  {path}")

print(f"\nResolved {len(repo_paths)} repo subfolders under {RQ2_ROOT}")


Already on 'master'


Your branch is up to date with 'origin/master'.
  T01_SHARED          /home/azureuser/repos/RQ2_T01_SHARED  (3320877 doc 8 reprecompute)


Already on 'master'


Your branch is up to date with 'origin/master'.
  T02_DATA_LOADER     /home/azureuser/repos/RQ2_T02_DATA_LOADER  (ebc09e2 before t04 precompute)


Already on 'master'


Your branch is up to date with 'origin/master'.
  T05_PAGEINDEX       /home/azureuser/repos/RQ2_T05_ARM2_PAGEINDEX  (93ca4ed fresh run ready)

Cloned/updated 3 repos in /home/azureuser/repos


*— cell 8 —*

## 4. Install Python packages

Editable installs of T01, T02, T05. T03/T04 are deliberately excluded -
they are only needed locally by `build_tree.py`, not at Azure runtime,
since the tree is shipped pre-built in the bundle.

In [6]:
# === cell 9 ===
import subprocess
import sys

PIP = [sys.executable, "-m", "pip", "install", "-q"]

base_pkgs = [
    "numpy>=1.26", "pandas>=2.0", "tqdm>=4.66", "ollama>=0.2", "PyMuPDF",
    "azure-storage-blob",
]
subprocess.run(PIP + base_pkgs, check=True)

for label, target in repo_paths.items():
    subprocess.run(PIP + ["-e", str(target)], check=True)
    print(f"  installed {label} from {target}")

# Defensive: PEP 660 editable installs occasionally fail to register a .pth
# in this Anaconda env without raising (`pip show` succeeds, but the
# package is missing from sys.path). Inject src/ paths so the package
# imports either way.
for label, target in repo_paths.items():
    src_path = str(target / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

print("\nAll packages installed (+ src/ paths added as fallback).")


  installed T01_SHARED from /home/azureuser/repos/RQ2_T01_SHARED
  installed T02_DATA_LOADER from /home/azureuser/repos/RQ2_T02_DATA_LOADER
  installed T05_PAGEINDEX from /home/azureuser/repos/RQ2_T05_ARM2_PAGEINDEX

All packages installed (+ src/ paths added as fallback).


*— cell 10 —*

## 5b. Fresh-run guard

`run_subset` is idempotent on `out_dir` — it reads any per-query JSON
already on disk and skips that LLM call. That's the right default for
resuming an interrupted run, but it silently makes a CODE-CHANGE re-run
reuse the OLD results.

Set `FRESH_RUN = True` to wipe both the VM-local bundle dir
(`/home/azureuser/bundle/<RUN_NAME>/`) and results dir
(`/home/azureuser/results/<RUN_NAME>/`) for THIS `RUN_NAME` only.
Other PDFs' caches on the same VM are untouched. Default `False` so
ordinary resumes still work.

In [7]:
# === cell 11 ===
import shutil
from pathlib import Path

from azure.storage.blob import ContainerClient

FRESH_RUN = True   # FLIP THIS WHEN RE-RUNNING WITH PATCHED CODE.
#FRESH_RUN = False   


if FRESH_RUN:
    # 1. Local VM dirs.
    for base in (BUNDLE_DIR, RESULTS_DIR):
        p = Path(base) / RUN_NAME
        if p.exists():
            print(f"Wiping {p}")
            shutil.rmtree(p)
        else:
            print(f"Already absent: {p}")
    # 2. Azure blob prefix. Cell 21's resume step pulls anything under
    #    t05_results/<RUN_NAME>/ back into the local results dir, which
    #    would silently re-cache stale per-query JSONs and make
    #    run_subset skip every query. Cleaning the blob here is the only
    #    way to guarantee a truly fresh run.
    _container = ContainerClient.from_container_url(AZURE_CONTAINER_SAS_URL)
    _prefix = f"{RESULTS_BLOB_PREFIX}/"
    _stale = [b.name for b in _container.list_blobs(name_starts_with=_prefix)]
    # Defensive: every name we delete must live under our prefix.
    assert all(n.startswith(_prefix) for n in _stale), \
        f"refusing — found blob outside {_prefix}"
    for name in _stale:
        _container.delete_blob(name)
    print(f"Cleared {len(_stale)} stale blobs from {_prefix}")
    print("\nFresh-run wipe complete.")
else:
    print("FRESH_RUN=False — keeping any existing local + blob results for"
          f" {RUN_NAME}. run_subset will resume from cached per-query JSONs.")


Wiping /home/azureuser/bundle/t04_2003_07_17_2013A31614
Wiping /home/azureuser/results/t04_2003_07_17_2013A31614
Cleared 0 stale blobs from t05_results/t04_2003_07_17_2013A31614/

Fresh-run wipe complete.


*— cell 12 —*

## 5. Download bundle from Azure Blob

The bundle was built locally with `scripts/prepare_azure_bundle.py` and
uploaded to the container. It contains the trees, queries (with
question texts joined from `bsard_corpus.db`), and the original GT
file.

In [8]:
# === cell 13 ===
import zipfile
from pathlib import Path

from azure.storage.blob import ContainerClient

# One bundle dir per RUN_NAME — without this, reusing a VM that has run a
# different PDF will MIX trees/manifest from that prior PDF with the
# newly-downloaded bundle, and cell 13's `trees/doc_*.json` glob will
# pick up whichever trees Linux happens to list first.
bundle_dir = Path(BUNDLE_DIR) / RUN_NAME
bundle_dir.mkdir(parents=True, exist_ok=True)

container = ContainerClient.from_container_url(AZURE_CONTAINER_SAS_URL)
local_zip = bundle_dir / "bundle.zip"
if not local_zip.exists():
    print(f"Downloading {BUNDLE_BLOB_NAME} ...")
    with local_zip.open("wb") as f:
        f.write(
            container.get_blob_client(BUNDLE_BLOB_NAME)
            .download_blob().readall()
        )
print(f"Bundle zip: {local_zip} ({local_zip.stat().st_size/1024:.1f} KB)")

# Extract (idempotent — overwrites in place).
with zipfile.ZipFile(local_zip) as zf:
    zf.extractall(bundle_dir)

print("\nBundle contents:")
for p in sorted(bundle_dir.rglob("*")):
    if p.is_file() and p != local_zip:
        rel = p.relative_to(bundle_dir)
        print(f"  {str(rel):<40}  {p.stat().st_size:>9,} B")


Bundle zip: /home/azureuser/bundle/t04_2003_07_17_2013A31614/bundle.zip (126.3 KB)

Bundle contents:
  gt.json                                       4,305 B
  manifest.json                                   718 B
  queries.json                                 17,957 B
  trees/doc_2003_07_17_2013A31614.json        623,776 B


*— cell 14 —*

## 6. Load tree, queries, manifest

In [9]:
# === cell 15 ===
import json
from pathlib import Path

bundle_dir = Path(BUNDLE_DIR) / RUN_NAME

# Cell 9's defensive sys.path hook makes this import safe even when the
# editable install didn't register cleanly.
from arm2_pageindex.tree_builder import compose_corpus_tree, load_law_tree

queries = json.loads((bundle_dir / "queries.json").read_text(encoding="utf-8"))
gt = json.loads((bundle_dir / "gt.json").read_text(encoding="utf-8"))
manifest_in = json.loads((bundle_dir / "manifest.json").read_text(encoding="utf-8"))

# Hard fail if the bundle's manifest doesn't match RUN_NAME — this is the
# canary for a stale bundle / wrong-PDF mix-up.
if manifest_in.get("run_name") not in (RUN_NAME, None):
    raise RuntimeError(
        f"Bundle manifest run_name={manifest_in.get('run_name')!r} "
        f"does not match RUN_NAME={RUN_NAME!r}. "
        f"Wipe {bundle_dir} and re-run cell 11."
    )

law_subtrees = []
for tp in sorted((bundle_dir / "trees").glob("doc_*.json")):
    node, m = load_law_tree(tp)
    law_subtrees.append(node)
    print(f"  {tp.name}  doc_id={m.get('document_id')}  "
          f"arts={m.get('n_articles')}  chaps={m.get('n_chapters')}  "
          f"derivable={m.get('chapter_derivable')}")

# When more than one law tree is loaded, wrap them under a corpus root so
# the navigator's law-selection step is meaningful.
tree = compose_corpus_tree(law_subtrees) if len(law_subtrees) > 1 else law_subtrees[0]

print(
    f"\nQueries: {len(queries)}  Trees: {len(law_subtrees)}  "
    f"GT entries: {len(gt)}  Run name: {manifest_in.get('run_name')!r}"
)


  doc_2003_07_17_2013A31614.json  doc_id=None  arts=257  chaps=92  derivable=True

Queries: 133  Trees: 1  GT entries: 133  Run name: 't04_2003_07_17_2013A31614'


*— cell 16 —*

## 7. Pre-flight smokes

Three gates before the long run.

1. **LLM warmup** — 3 cold calls; drop call 1, average 2-3 as the
   "warm" baseline tok/s.
2. **Single-query end-to-end** — full navigation on the first query;
   confirms French JSON-strict parsing.
3. **5-question pilot** — small batch with parse-fail rate +
   LLM-calls-per-query distribution + extrapolated ETA.

In [10]:
import time

from shared.llm import LLMClient

llm = LLMClient(model=OLLAMA_MODEL)
llm._options["num_ctx"] = 16384   # workaround: T01 LLMClient on the VM is pre-fix
print(f"LLM options: {llm._options}")


LLM options: {'temperature': 0.0, 'num_ctx': 16384}


In [11]:
# # === cell 17 ===
# import time

# from shared.llm import LLMClient

# # num_ctx=16384 overrides Ollama's 4k default. T05 chapter-selection
# # prompts are 6-9k tokens on the curated 5 PDFs and were silently
# # truncated at 4k pre-fix, erasing the chapter list (see CHANGE_NOTES).
# llm = LLMClient(model=OLLAMA_MODEL, num_ctx=16384)

# warmup_prompt = (
#     "Réponds par un seul mot en JSON : {\"reply\": \"oui\"}\n"
#     "Question : Le ciel est-il bleu ?"
# )

# latencies = []
# for i in range(3):
#     t0 = time.perf_counter()
#     r = llm.generate(warmup_prompt)
#     dt = (time.perf_counter() - t0) * 1000
#     latencies.append(dt)
#     tps = r.output_tokens / max(r.latency_ms / 1000, 0.001)
#     print(
#         f"call {i+1}: {dt:>7.0f} ms  "
#         f"in={r.input_tokens}  out={r.output_tokens}  ~{tps:.1f} tok/s  "
#         f"text={r.text!r}"
#     )

# warm_mean_ms = sum(latencies[1:]) / max(len(latencies) - 1, 1)
# print(f"\nWarm baseline (calls 2-3 mean): {warm_mean_ms:.0f} ms / call")


In [12]:
# === cell 18 ===
import time

from arm2_pageindex.navigator import NavigatorConfig
from arm2_pageindex.retriever import run_arm2b

cfg = NavigatorConfig(max_iterations=2)
smoke_q = queries[0]
print(f"Smoke query (qid={smoke_q['query_id']}): {smoke_q['query_text']!r}\n")

t0 = time.perf_counter()
result = run_arm2b(
    query=smoke_q["query_text"],
    query_id=smoke_q["query_id"],
    tree=tree,
    llm=llm,
    cfg=cfg,
)
elapsed = time.perf_counter() - t0

print(f"Wall time:       {elapsed:.1f} s")
print(f"LLM calls:       {result.cost.get('llm_calls')}")
print(f"Tokens in/out:   {result.cost.get('tokens_in')} / {result.cost.get('tokens_out')}")
print(f"Parse failures:  {result.cost.get('parse_failures')}")
print(f"Exit reason:     {result.cost.get('exit_reason')}")
print(f"Ranked:          {len(result.ranked_items)}")
for item in result.ranked_items[:5]:
    md_ = item["metadata"]
    print(
        f"  score={item['score']:>4.1f}  "
        f"bsard_id={md_.get('bsard_id')}  "
        f"art_no={md_.get('article_number')!r}  {item['id']}"
    )

print("\nSteps:")
for s in (result.trace or {}).get("steps", []):
    name = s.get("step", "?")
    if s.get("skipped"):
        print(f"  [SKIP]   {name}  reason={s.get('reason')}")
    else:
        ok = "ok" if s.get("parse_ok") else "FAIL"
        print(
            f"  parse={ok:<4}  {name:<38}  "
            f"in={s.get('tokens_in', 0):>5}  out={s.get('tokens_out', 0):>5}  "
            f"{s.get('latency_ms', 0):>7.1f} ms"
        )


Smoke query (qid=28): 'Qui doit payer le précompte immobilier ?'

Wall time:       92.4 s
LLM calls:       4
Tokens in/out:   10389 / 288
Parse failures:  0
Exit reason:     sufficient
Ranked:          100
  score= 3.0  bsard_id=844  art_no='226'  ART_doc_8_bsard_844
  score= 1.0  bsard_id=746  art_no='136.Les'  ART_doc_8_bsard_746
  score= 1.0  bsard_id=747  art_no='137'  ART_doc_8_bsard_747
  score= 0.0  bsard_id=745  art_no='135'  ART_doc_8_bsard_745
  score= 0.0  bsard_id=610  art_no='1-2'  ART_doc_8_bsard_610

Steps:
  [SKIP]   law_selection  reason=single_doc_corpus
  parse=ok    chapter_selection[LAW_8]                in= 8393  out=   50  84484.4 ms
  parse=ok    article_selection[LAW_8_CH_TITRE_V_DE_LA_TUTELLE_ET_DES_MODES_DE_CO]  in=  669  out=   95   3019.1 ms
  parse=ok    article_selection[LAW_8_CH_Art_226_1_Precompte_immobilier]  in=  444  out=   66   2120.0 ms
  parse=ok    evaluate[iter=0]                        in=  883  out=   77   2802.7 ms


In [13]:
# === cell 19 ===
import statistics

from arm2_pageindex.pipeline import run_subset

pilot_queries = queries[: min(5, len(queries))]

pilot_results = run_subset(
    tree=tree,
    queries=pilot_queries,
    llm=llm,
    cfg=cfg,
    out_dir=None,            # in-memory only — do not pollute results dir yet
    progress=False,
    log_every=1,
)

call_counts = [r.cost.get("llm_calls", 0) for r in pilot_results]
latencies = [r.cost.get("latency_ms", 0) for r in pilot_results]
parse_fails = sum(r.cost.get("parse_failures", 0) for r in pilot_results)
total_calls = sum(call_counts)
exit_reasons = {}
for r in pilot_results:
    er = r.cost.get("exit_reason", "?")
    exit_reasons[er] = exit_reasons.get(er, 0) + 1

print(f"Pilot ({len(pilot_results)} queries):")
print(
    f"  LLM calls/q (mean / median / max): "
    f"{statistics.mean(call_counts):.1f} / "
    f"{int(statistics.median(call_counts))} / {max(call_counts)}"
)
print(
    f"  Latency ms/q (mean / max):         "
    f"{statistics.mean(latencies):.0f} / {max(latencies):.0f}"
)
print(
    f"  Parse failures total:              "
    f"{parse_fails} / {total_calls} calls "
    f"({100 * parse_fails / max(total_calls, 1):.1f}%)"
)
print(f"  Exit reasons:                      {exit_reasons}")


Pilot (5 queries):
  LLM calls/q (mean / median / max): 5.4 / 5 / 7
  Latency ms/q (mean / max):         24000 / 27321
  Parse failures total:              0 / 27 calls (0.0%)
  Exit reasons:                      {'sufficient': 5}


*— cell 20 —*

### Time-budget gate

Inspect the projected total below. **Continue past this cell only when
the numbers look acceptable.** Cell 14 starts the full run.

In [14]:
# === cell 21 ===
import statistics

mean_latency_s = statistics.mean(latencies) / 1000.0
mean_calls = statistics.mean(call_counts)
total_q = len(queries)

projected_s = mean_latency_s * total_q
projected_calls = mean_calls * total_q
projected_fails = (parse_fails / max(total_calls, 1)) * projected_calls

print(f"Projected full run on {total_q} queries:")
print(f"  Wall time:   ~{projected_s/60:.1f} min  ({projected_s:.0f} s)")
print(f"  LLM calls:   ~{projected_calls:.0f}")
print(f"  Parse fails: ~{projected_fails:.0f}")
print()
print("Inspect — then run the next cell to start the long run.")


Projected full run on 133 queries:
  Wall time:   ~53.2 min  (3192 s)
  LLM calls:   ~718
  Parse fails: ~0

Inspect — then run the next cell to start the long run.


*— cell 22 —*

## 8. Full run

Resume-friendly: any per-query JSON already in `RESULTS_DIR/<run>/`
short-circuits in `run_subset`. The cell below also pre-pulls any
prior results from the blob so a fresh kernel restart picks up where
the previous one left off.

In [15]:
# from arm2_pageindex.navigator import NavigatorConfig
# cfg = NavigatorConfig(max_iterations=2)


In [16]:
# === cell 23 ===
from pathlib import Path

from arm2_pageindex.pipeline import run_subset

run_name = manifest_in.get("run_name", "default")
results_dir = Path(RESULTS_DIR) / run_name
results_dir.mkdir(parents=True, exist_ok=True)

# Resume: pull any prior per-query JSONs from blob.
prefix = f"{RESULTS_BLOB_PREFIX}/"
n_resumed = 0
for blob in container.list_blobs(name_starts_with=prefix):
    name = blob.name[len(prefix):]
    target = results_dir / name
    if target.exists():
        continue
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.open("wb") as f:
        f.write(container.get_blob_client(blob).download_blob().readall())
    n_resumed += 1
if n_resumed:
    print(f"Resumed: pulled {n_resumed} prior per-query JSONs from blob.")

results = run_subset(
    tree=tree,
    queries=queries,
    llm=llm,
    cfg=cfg,
    out_dir=results_dir,
    progress=True,
    log_every=10,
)
print(f"\nDone — {len(results)} results in {results_dir}")


T05: 100%|██████████| 133/133 [1:02:32<00:00, 28.22s/it]


Done — 133 results in /home/azureuser/results/t04_2003_07_17_2013A31614


*— cell 24 —*

## 9. Upload results to Blob

In [17]:
# === cell 25 ===
n_uploaded = 0
for path in sorted(results_dir.iterdir()):
    if not path.is_file():
        continue
    blob_name = f"{RESULTS_BLOB_PREFIX}/{path.name}"
    with path.open("rb") as f:
        container.get_blob_client(blob_name).upload_blob(f, overwrite=True)
    n_uploaded += 1

print(f"Uploaded {n_uploaded} files to {RESULTS_BLOB_PREFIX}/")


Uploaded 134 files to t05_results/t04_2003_07_17_2013A31614/


*— cell 26 —*

### Stage results for copy back to the local repo

Downloads per-query `q<qid>.json` files from blob into a directory on
the VM that mirrors the target layout in the local repo
(`RQ2_T05_ARM2_PAGEINDEX/data/<pdf-stem>/results/`). Independent of the
run — works on a fresh kernel that just has `container` and
`RESULTS_BLOB_PREFIX` defined. Set `PDF_STEM` per run.

In [18]:
# === cell 27 ===
# Mirror layout: <pdf-stem>/results/q<qid>.json so a single scp -r
# drops the files straight into RQ2_T05_ARM2_PAGEINDEX/data/.
import re
from pathlib import Path

PDF_STEM = "2004_05_27_2004A27101"           # edit per run
LOCAL_TARGET_REL = f"RQ2_T05_ARM2_PAGEINDEX/data/{PDF_STEM}/results"
EXPORT_ROOT = Path("/home/azureuser/local_export")
EXPORT_DIR = EXPORT_ROOT / PDF_STEM / "results"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

per_query_re = re.compile(r"^q\d+\.json$")
prefix = f"{RESULTS_BLOB_PREFIX}/"

downloaded = []
for blob in container.list_blobs(name_starts_with=prefix):
    name = blob.name[len(prefix):]
    if not per_query_re.match(name):
        continue
    target = EXPORT_DIR / name
    with target.open("wb") as f:
        f.write(container.get_blob_client(blob).download_blob().readall())
    downloaded.append(target)

print(f"Staged {len(downloaded)} per-query JSONs at {EXPORT_DIR}:")
for p in sorted(downloaded):
    print(f"  {p}  ({p.stat().st_size:,} B)")

print()
print(f"Local target in repo: {LOCAL_TARGET_REL}/")
print()
print("To copy them down (replace <vm-ip>):")
print(f"  scp -r azureuser@<vm-ip>:{EXPORT_ROOT}/{PDF_STEM} \\")
print(f'         "<repo-root>/RQ2_T05_ARM2_PAGEINDEX/data/"')
print()
print("Or browse via the Azure portal file explorer.")


Staged 133 per-query JSONs at /home/azureuser/local_export/2004_05_27_2004A27101/results:
  /home/azureuser/local_export/2004_05_27_2004A27101/results/q1000.json  (518,269 B)
  /home/azureuser/local_export/2004_05_27_2004A27101/results/q1001.json  (487,493 B)
  /home/azureuser/local_export/2004_05_27_2004A27101/results/q1002.json  (460,107 B)
  /home/azureuser/local_export/2004_05_27_2004A27101/results/q1003.json  (460,367 B)
  /home/azureuser/local_export/2004_05_27_2004A27101/results/q1005.json  (519,288 B)
  /home/azureuser/local_export/2004_05_27_2004A27101/results/q1006.json  (462,499 B)
  /home/azureuser/local_export/2004_05_27_2004A27101/results/q1007.json  (462,735 B)
  /home/azureuser/local_export/2004_05_27_2004A27101/results/q1008.json  (460,064 B)
  /home/azureuser/local_export/2004_05_27_2004A27101/results/q1009.json  (469,852 B)
  /home/azureuser/local_export/2004_05_27_2004A27101/results/q1010.json  (472,159 B)
  /home/azureuser/local_export/2004_05_27_2004A27101/results

*— cell 28 —*

## 10. Cleanup\n\nKill the Ollama daemon to release GPU memory.

In [19]:
# === cell 29 ===
import subprocess
subprocess.run(["pkill", "-f", "ollama"], check=False)
print("Ollama stopped.")


Ollama stopped.


pkill: killing pid 650 failed: Operation not permitted
pkill: killing pid 3806 failed: Operation not permitted
